# 05 — Excel/CSV Business Summary
    Use pandas to create a workbook of stakeholder-friendly summary tables that can be opened in Excel.

In [ ]:
from pathlib import Path
    import pandas as pd
    DATA = Path("../01_Raw_Data")
    emp = pd.read_csv(DATA/"employees.csv")
    rec = pd.read_csv(DATA/"recruitment_pipeline.csv")
    train = pd.read_csv(DATA/"training_records.csv")
    dept = emp.groupby("department").agg(
        headcount=("employee_id","count"),
        attrition_rate_pct=("attrition_flag",lambda x: round(x.mean()*100,2)),
        avg_salary_inr=("annual_salary_inr","mean"),
        avg_engagement=("engagement_score","mean")).reset_index()
    hiring = rec.groupby("candidate_source").agg(
        applications=("application_id","count"),
        hires=("current_stage",lambda x:(x=="Hired").sum())).reset_index()
    hiring["hire_conversion_pct"]=(100*hiring.hires/hiring.applications).round(2)
    learning = train.groupby("course_category").agg(
        enrollments=("training_id","count"),
        completion_rate_pct=("completion_status",lambda x: round((x=="Completed").mean()*100,2)),
        average_score=("assessment_score","mean")).reset_index()
    with pd.ExcelWriter("../02_Excel_Analysis/PeoplePulse_Management_Summary.xlsx", engine="openpyxl") as writer:
        dept.to_excel(writer, sheet_name="Department KPIs", index=False)
        hiring.to_excel(writer, sheet_name="Recruitment", index=False)
        learning.to_excel(writer, sheet_name="Learning", index=False)
    print("Workbook created.")